In [0]:
''' 
%sql
CREATE CATALOG IF NOT EXISTS Lakeflow_PL_cat;
CREATE SCHEMA IF NOT EXISTS Lakeflow_PL_cat.Lakeflow_PL_sch;
CREATE VOLUME IF NOT EXISTS Lakeflow_PL_cat.Lakeflow_PL_sch.cloud_datalake;
CREATE VOLUME IF NOT EXISTS Lakeflow_PL_cat.Lakeflow_PL_sch.bronze;
CREATE VOLUME IF NOT EXISTS Lakeflow_PL_cat.Lakeflow_PL_sch.silver;
CREATE VOLUME IF NOT EXISTS Lakeflow_PL_cat.Lakeflow_PL_sch.gold;
'''

In [0]:
cloudsrc="/Volumes/lakeflow_pl_cat/lakeflow_pl_sch/cloud_datalake/"
bronzetgt ="/Volumes/lakeflow_pl_cat/lakeflow_pl_sch/bronze/"
chkptlocation="/Volumes/lakeflow_pl_cat/lakeflow_pl_sch/cloud_datalake/ckpt/_checkpoint"
schemalocation ="/Volumes/lakeflow_pl_cat/lakeflow_pl_sch/cloud_datalake/schemaL/_schema"
#it is not mandate to have checkpointlocation in read section, however write will have it and is taken care while writing the data to filter out only incremented data copy. 
df1= spark.readStream.format("cloudfiles")\
    .option("cloudfiles.format","csv")\
    .option("cloudfiles.maxFilesPerTrigger",1)\
    .option("cloudfiles.inferSchema",True)\
    .option("header",True)\
    .option("cloudfiles.schemaEvolutionMode","addNewColumns")\
    .option("checkpointLocation",chkptlocation)\
    .option("cloudfiles.schemaLocation",schemalocation)\
    .load(cloudsrc)    
#.option("cloudFiles.useNotifications", "true") (Remove this option to enable directory listing)
#maxFilesPerTrigger - this property help spark to process howmany files in an iteration to control the resource utilization (all files will be processed ultimately)
#in cloud_datalake: AL1 file is initial ingestion, AL2 file is following ingestion, AL3 file is schema change ingestion where mergeSchema is included in write code and schemaevolutionMode in read code.

In [0]:
df1.writeStream.trigger(availableNow=True)\
.option("checkpointLocation", chkptlocation)\
.option("cloudFiles.schemaLocation", schemalocation)\
.option("mergeSchema", "true") \
.start(bronzetgt) # Ensure bronzetgt is a directory, not a file
    #.option("mergeSchema", True) - included only when schema evolution happens.

In [0]:
df_brnz=spark.read.format("delta").load(bronzetgt).orderBy("city_name").show(100)
